# **FAERS - FEATURE ENGINEERING**

# 1 - Load Data

In [ ]:
# load packages
import pandas as pd
import numpy as np
from collections import Counter

In [ ]:
# load data
df = pd.read_parquet("/Users/rakshabasnet/Downloads/FAERS_final.parquet")

# subset for testing
df = df.head(50_000) # remove when we test on full dataset
print(f"Subset shape: {df.shape}")
print(df.head(5))

# 2 - Imputation

In [ ]:
# fill na's
df["sex_bin"] = df["sex_bin"].fillna("Unknown")
df["wt_kg"] = df["wt_kg"].fillna(-1)
df["age_years"] = df["age_years"].fillna(-1)

def to_list_safe(x):
    if isinstance(x, list):
        return x
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ["Unknown"]
    if isinstance(x, np.ndarray):
        return x.tolist()
    return [x]

df["rx_name"] = df["rx_name"].apply(to_list_safe)
df["reactions"] = df["reactions"].apply(to_list_safe)

# unknown indicators - creating "unknown" cols to specify whether there is missing value for that specific col
def is_all_unknown(lst):
    for v in lst:
        if v is None or (isinstance(v, float) and np.isnan(v)) or v == "Unknown":
            continue
        return 0
    return 1

df["sex_unknown"] = (df["sex_bin"] == "Unknown").astype(int)
df["wt_unknown"] = (df["wt_kg"] == -1).astype(int)
df["age_unknown"] = (df["age_years"] == -1).astype(int)
df["rx_unknown"] = df["rx_name"].apply(is_all_unknown)
df["reacs_unknown"] = df["reactions"].apply(is_all_unknown)

# fill outcomes na's
outcome_cols = [
    "outcome_death",
    "outcome_life_threatening",
    "outcome_hospitalisation",
    "outcome_disability",
    "outcome_congenital_anomaly",
    "outcome_intervention_required",
    "outcome_other_serious"
]
df[outcome_cols] = df[outcome_cols].fillna(False)

# 3 - Filter rx_name & reactions 

In [ ]:
# top drugs
all_drugs = [r for sub in df["rx_name"] for r in sub if r != "Unknown"]
drug_counts = Counter(all_drugs)
top_550_drugs = [r for r, _ in drug_counts.most_common(550)]
print(top_550_drugs)

# top reactions
all_reacs = [r for sub in df["reactions"] for r in sub if r != "Unknown"]
reac_counts = Counter(all_reacs)
top_300_reacs = [r for r, _ in reac_counts.most_common(300)]
print(top_300_reacs)

In [ ]:
# quick check to get count of all_reacs or all_drugs
Counter(all_reacs).most_common(10)

# 4 - Record level multi-hot encoding

In [ ]:
# multi-hot encoding by the record
def encode_list(lst, top_set):
    return {k: int(k in lst) for k in top_set}

rx_mh_list = df["rx_name"].apply(lambda lst: encode_list(lst, top_550_drugs))
reac_mh_list = df["reactions"].apply(lambda lst: encode_list(lst, top_300_reacs))

rx_mh = pd.DataFrame(rx_mh_list.tolist(), index=df.index)
reac_mh = pd.DataFrame(reac_mh_list.tolist(), index=df.index)

#print(rx_mh.head(30))
#print(reac_mh.head(30))

# 5 - Groupby caseid

In [ ]:
# other_rx / other_reacs 
df["other_rx"] = df["rx_name"].apply(lambda lst: int(all(r not in top_550_drugs for r in lst if r != "Unknown")))
df["other_reacs"] = df["reactions"].apply(lambda lst: int(all(r not in top_300_reacs for r in lst if r != "Unknown")))

# groupby caseid
group_cols = ["sex_bin","sex_unknown","age_years","age_unknown","wt_kg","wt_unknown", "rx_name","rx_unknown","other_rx","reactions","reacs_unknown","other_reacs"] + outcome_cols
df_grouped = df.groupby("caseid")[group_cols].agg(list)

# 6 - Case-level unknowns / “other” flags computation

In [ ]:
# recompute unknowns / other flags at case level
df_grouped["sex_unknown"] = df_grouped["sex_bin"].apply(lambda x: int(all(v=="Unknown" for v in x)))
df_grouped["age_unknown"] = df_grouped["age_years"].apply(lambda x: int(all(v==-1 for v in x)))
df_grouped["wt_unknown"] = df_grouped["wt_kg"].apply(lambda x: int(all(v==-1 for v in x)))
df_grouped["rx_unknown"] = df_grouped["rx_name"].apply(lambda lsts: int(any("Unknown" in sub or len(sub)==0 for sub in lsts)))
df_grouped["reacs_unknown"] = df_grouped["reactions"].apply(lambda lsts: int(any("Unknown" in sub or len(sub)==0 for sub in lsts)))
df_grouped["other_rx"] = df_grouped["other_rx"].apply(lambda x: int(all(v==1 for v in x)))
# set other_reacs = 0 if patient has any top 300 reaction
df_grouped["other_reacs"] = df_grouped["reactions"].apply(lambda lsts: int(all(all(r not in top_300_reacs for r in sub if r != "Unknown") for sub in lsts)))

# 7 - Case level multi-hot

In [ ]:
# flatten lists for manual multi-hot
def flatten_and_filter(lst_of_lsts, top_set):
    flat = []
    for sub in lst_of_lsts:
        sublst = sub if isinstance(sub, list) else [sub]
        flat.extend([r for r in sublst if r in top_set])
    return flat

df_grouped["rx_name"] = df_grouped["rx_name"].apply(lambda x: flatten_and_filter(x, top_550_drugs))
df_grouped["reactions"] = df_grouped["reactions"].apply(lambda x: flatten_and_filter(x, top_300_reacs))

# multi-hot encoding at case level
def multi_hot_encode(lst, top_set):
    return {k: int(k in lst) for k in top_set}

rx_mh_case = pd.DataFrame(df_grouped["rx_name"].apply(lambda lst: multi_hot_encode(lst, top_550_drugs)).tolist(), index=df_grouped.index)
reac_mh_case = pd.DataFrame(df_grouped["reactions"].apply(lambda lst: multi_hot_encode(lst, top_300_reacs)).tolist(), index=df_grouped.index)

# 8 - Concatenate into final df

In [ ]:
# drop original columns and concatenate
df_grouped = df_grouped.drop(columns=["rx_name","reactions"])
df_grouped = pd.concat([df_grouped, rx_mh_case, reac_mh_case], axis=1)

print("Final df_grouped shape:", df_grouped.shape)
print(df_grouped.head())

## Validating

In [ ]:
# double check na count
total_na = df_grouped.isna().sum().sum()
print("total na in all cols:", total_na)

In [ ]:
# sum of all ones in rx_names
rx_ones_count = rx_mh_case.sum()
print(rx_ones_count)

In [ ]:
# count of all zeros in rx_names
rx_zeros_count = (rx_mh_case == 0).sum()
print(rx_zeros_count)

In [ ]:
# sum of all ones in reactions
reacs_ones_count = reac_mh_case.sum()
print(reacs_ones_count)

In [ ]:
# count of all zeros in reactions
reacs_zeros_count = (reac_mh_case == 0).sum()
print(reacs_zeros_count)